In [0]:
# DataSource: Base class for different data reading strategies
# Uses Factory pattern to create appropriate reader based on file format
class DataSource:
    

    def __init__(self, path, schema=None):
        # path: File path or table name to read from
        # schema: Optional schema definition (primarily used for CSV)
        self.path = path
        self.schema = schema
    
    def get_data_frame(self):
        # Abstract method - must be implemented by subclasses
        raise NotImplementedError("Not implemented")


# Concrete implementation for reading CSV files
class CSVDataSource(DataSource):
    

    def get_data_frame(self):
        # Read CSV file with header row and apply provided schema
        # mode='permissive': Sets null for corrupted records
        return (
            spark
            .read
            .format('csv')
            .option('header', True)
            .option('mode', 'permissive')
            .schema(self.schema)
            .load(self.path)
        )


# Note:
# Parquet / ORC / Delta formats already store schema metadata internally,
# so explicit schema definition is usually not required.

# Concrete implementation for reading Parquet files
class ParquetDataSource(DataSource):
    """
    Concrete DataSource implementation for Parquet files.
    """

    def get_data_frame(self):
        """
        Read Parquet file into a Spark DataFrame.
        """
        # Read Parquet file - schema is inferred from file metadata
        return (
            spark
            .read
            .format('parquet')
            .option('mode', 'permissive')
            .load(self.path)
        )


# Concrete implementation for reading ORC files
class ORCDataSource(DataSource):
    """
    Concrete DataSource implementation for ORC files.
    """

    def get_data_frame(self):
        """
        Read ORC file into a Spark DataFrame.
        """
        # Read ORC file - schema is inferred from file metadata
        return (
            spark
            .read
            .format('orc')
            .option('mode', 'permissive')
            .load(self.path)
        )


# Concrete implementation for reading Delta tables
class DeltaDataSource(DataSource):
    """
    Concrete DataSource implementation for Delta tables.

    Note:
    - Path is treated as a table name registered in the metastore
    """

    def get_data_frame(self):
        """
        Read Delta table into a Spark DataFrame using table name.
        """
        # Path represents table name in metastore, not file path
        table_name = self.path
        return (
            spark
            .read
            .table(table_name)
        )


# Factory function: Returns appropriate DataSource instance based on data_type
def get_data_source(data_type, file_path, schema=None):
    """
    Factory method to return the appropriate DataSource implementation.

    Parameters:
    - data_type : Type of source ('csv', 'parquet', 'orc', 'delta')
    - file_path : File path or table name
    - schema    : Optional schema (used only for CSV)

    Returns:
    - Instance of a DataSource subclass
    """

    if data_type == 'csv':
        # CSV reader with optional schema
        return CSVDataSource(file_path, schema)

    elif data_type == 'parquet':
        # Parquet reader - schema inferred automatically
        return ParquetDataSource(file_path)

    elif data_type == 'orc':
        # ORC reader - schema inferred automatically
        return ORCDataSource(file_path)

    elif data_type == 'delta':
        # Delta table reader using table name
        return DeltaDataSource(file_path)

    else:
        # Invalid data source type provided
        raise Exception(f"Invalid data type: {data_type}")